In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import os

# Đường dẫn gốc
base_path = '/content/drive/MyDrive/Data clean'
categories = ['Chó', 'Mèo']
image_extensions = ('.png', '.jpg', '.jpeg', '.webp', '.bmp', '.img')

print(f"--- Thống kê ảnh trong: {base_path} ---")

total_images = 0
for cat in categories:
    cat_path = os.path.join(base_path, cat)
    if os.path.exists(cat_path):
        # Đếm các file có đuôi nằm trong danh sách hỗ trợ
        files = [f for f in os.listdir(cat_path) if f.lower().endswith(image_extensions)]
        count = len(files)
        total_images += count
        print(f"+ Thư mục '{cat}': {count} ảnh")
    else:
        print(f"! Không tìm thấy thư mục: {cat_path}")

print(f"\n=> Tổng cộng: {total_images} ảnh")

--- Thống kê ảnh trong: /content/drive/MyDrive/Data clean ---
+ Thư mục 'Chó': 1023 ảnh
+ Thư mục 'Mèo': 1006 ảnh

=> Tổng cộng: 2029 ảnh


In [ ]:
import os
from PIL import Image
!pip install pillow-heif
from pillow_heif import register_heif_opener

# Register HEIF opener for .img files
register_heif_opener()

base_path = '/content/drive/MyDrive/Data clean'
categories = ['Chó', 'Mèo']
image_extensions = ('.png', '.jpg', '.jpeg', '.webp', '.bmp', '.img')

print(" Starting image format conversion to .jpg...")

for cat in categories:
    cat_dir = os.path.join(base_path, cat)
    if not os.path.exists(cat_dir):
        continue

    files = [f for f in os.listdir(cat_dir) if f.lower().endswith(image_extensions)]
    print(f"\nProcessing folder: {cat} ({len(files)} files)")

    converted_count = 0
    for filename in files:
        file_path = os.path.join(cat_dir, filename)
        name_no_ext = os.path.splitext(filename)[0]
        new_file_path = os.path.join(cat_dir, name_no_ext + '.jpg')

        try:
            with Image.open(file_path) as img:
                # Convert to RGB (required for JPG)
                rgb_img = img.convert('RGB')
                rgb_img.save(new_file_path, 'JPEG', quality=95)

            # If the original file wasn't already a .jpg, delete the old one
            if file_path.lower() != new_file_path.lower():
                os.remove(file_path)

            converted_count += 1
            if converted_count % 100 == 0:
                print(f"... Converted {converted_count} images")
        except Exception as e:
            print(f" Error converting {filename}: {e}")

print(f"\n Conversion finished! All images in '{base_path}' are now .jpg.")

In [5]:
import os
import io
import time
import ipywidgets as widgets
from IPython.display import display
from PIL import Image, ImageDraw
from google.colab import output

# --- Cấu hình ---
BASE_DIR = '/content/drive/MyDrive/Data clean'
CATEGORIES_MAP = {'Chó': 0, 'Mèo': 1}
VALID_EXTENSIONS = ('.jpg', '.jpeg', '.png')
SAVE_DIR = '/content/labels'

os.makedirs(SAVE_DIR, exist_ok=True)

class FullManualLabeler:
    def __init__(self, img_base_dir, categories_map):
        self.all_images = []
        self.current_idx = 0
        self.points = []
        self.categories_map = categories_map

        for cat_name, class_id in categories_map.items():
            img_dir = os.path.join(img_base_dir, cat_name)
            os.makedirs(os.path.join(SAVE_DIR, cat_name), exist_ok=True)
            if os.path.exists(img_dir):
                files = sorted([f for f in os.listdir(img_dir) if f.lower().endswith(VALID_EXTENSIONS)])
                for f in files:
                    self.all_images.append({
                        'path': os.path.join(img_dir, f),
                        'suggested_id': class_id,
                        'name': f,
                        'category': cat_name
                    })

        self.img_widget = widgets.Image(format='jpeg')
        self.info = widgets.HTML("<h3>Sẵn sàng...</h3>")
        self.class_sel = widgets.Dropdown(
            options=[('Chó (0)', 0), ('Mèo (1)', 1)],
            description='Lớp:',
        )
        self.btn_reset = widgets.Button(description="Xóa nhãn/Vẽ lại", button_style='warning')
        self.btn_back = widgets.Button(description="Quay lại", button_style='danger')
        self.btn_next = widgets.Button(description="Bỏ qua/Tiếp", button_style='info')

        self.btn_reset.on_click(self.reset_points)
        self.btn_back.on_click(self.prev_img)
        self.btn_next.on_click(lambda x: self.next_img())

        self.layout = widgets.VBox([
            self.info,
            widgets.HTML("<p style='color:red;'><b>Click 2 điểm để tạo khung.</b> Tự chuyển sau 1 giây.</p>"),
            self.img_widget,
            widgets.HBox([self.class_sel, self.btn_back, self.btn_reset, self.btn_next])
        ], layout=widgets.Layout(align_items='center'))

        output.register_callback('notebook.on_click', self.on_img_click)
        output.register_callback('notebook.auto_next', self.next_img)

        if self.all_images:
            display(self.layout)
            # Tìm ảnh đầu tiên chưa có nhãn để bắt đầu
            self.skip_labeled_images()
            self.show_image()
            self.setup_click_event()
        else:
            print("Không tìm thấy ảnh!")

    def setup_click_event(self):
        js = """
        (function() {
            const findAndBind = () => {
                const imgs = document.querySelectorAll('.jupyter-widgets.widget-image');
                const target = imgs[imgs.length - 1];
                if (target) {
                    target.onclick = (e) => {
                        const rect = target.getBoundingClientRect();
                        google.colab.kernel.invokeFunction('notebook.on_click', [e.clientX - rect.left, e.clientY - rect.top], {});
                    };
                    target.style.cursor = 'crosshair';
                }
            };
            findAndBind();
            setTimeout(findAndBind, 200);
        })();
        """
        display(widgets.HTML(f"<script>{js}</script>"))

    def has_label(self, idx):
        data = self.all_images[idx]
        save_path = os.path.join(SAVE_DIR, data['category'], os.path.splitext(data['name'])[0] + ".txt")
        return os.path.exists(save_path)

    def skip_labeled_images(self):
        while self.current_idx < len(self.all_images) and self.has_label(self.current_idx):
            self.current_idx += 1

    def get_existing_box(self, data):
        save_path = os.path.join(SAVE_DIR, data['category'], os.path.splitext(data['name'])[0] + ".txt")
        if os.path.exists(save_path):
            with open(save_path, 'r') as f:
                content = f.read().strip().split()
                if len(content) == 5:
                    _, xc, yc, w, h = map(float, content)
                    l = (xc - w/2) * self.disp_w
                    t = (yc - h/2) * self.disp_h
                    r = (xc + w/2) * self.disp_w
                    b = (yc + h/2) * self.disp_h
                    return [l, t, r, b]
        return None

    def show_image(self, box=None):
        if self.current_idx >= len(self.all_images):
            self.info.value = "<h2> Đã hoàn thành tất cả!</h2>"
            return

        data = self.all_images[self.current_idx]
        self.class_sel.value = data['suggested_id']
        self.raw_img = Image.open(data['path']).convert('RGB')
        self.raw_img.thumbnail((600, 600))
        self.disp_w, self.disp_h = self.raw_img.size

        if box is None: box = self.get_existing_box(data)

        draw_img = self.raw_img.copy()
        if box:
            d = ImageDraw.Draw(draw_img)
            d.rectangle(box, outline="red", width=3)

        buf = io.BytesIO()
        draw_img.save(buf, format='JPEG')
        self.img_widget.value = buf.getvalue()
        self.info.value = f"<h3>Ảnh {self.current_idx+1}/{len(self.all_images)}: {data['name']}</h3>"

    def on_img_click(self, x, y):
        self.points.append((x, y))
        if len(self.points) == 2:
            x1, y1 = self.points[0]
            x2, y2 = self.points[1]
            box = [min(x1, x2), min(y1, y2), max(x1, x2), max(y1, y2)]
            self.show_image(box=box)
            self.save_yolo(x1, y1, x2, y2)
        else:
            self.info.value += "<br><b style='color:blue;'>Điểm 1 OK. Click điểm 2...</b>"

    def save_yolo(self, x1, y1, x2, y2):
        data = self.all_images[self.current_idx]
        l, r, t, b = min(x1, x2), max(x1, x2), min(y1, y2), max(y1, y2)
        xc, yc = ((l + r) / 2) / self.disp_w, ((t + b) / 2) / self.disp_h
        w, h = (r - l) / self.disp_w, (b - t) / self.disp_h

        save_path = os.path.join(SAVE_DIR, data['category'], os.path.splitext(data['name'])[0] + ".txt")
        with open(save_path, 'w') as f:
            f.write(f"{self.class_sel.value} {xc:.6f} {yc:.6f} {w:.6f} {h:.6f}")

        self.points = []
        self.info.value += "<br><b style='color:green;'>Đã lưu! Chờ 1 giây để chuyển...</b>"
        js_next = "<script>setTimeout(() => { google.colab.kernel.invokeFunction('notebook.auto_next', [], {}); }, 1000);</script>"
        display(widgets.HTML(js_next))

    def reset_points(self, b):
        data = self.all_images[self.current_idx]
        save_path = os.path.join(SAVE_DIR, data['category'], os.path.splitext(data['name'])[0] + ".txt")
        if os.path.exists(save_path): os.remove(save_path)
        self.points = []
        self.show_image()

    def prev_img(self, b=None):
        if self.current_idx > 0:
            self.current_idx -= 1
            self.points = []
            self.show_image()
            self.setup_click_event()

    def next_img(self, b=None):
        if self.current_idx + 1 < len(self.all_images):
            self.current_idx += 1
            # Bỏ qua các ảnh đã có nhãn
            self.skip_labeled_images()
            self.points = []
            self.show_image()
            self.setup_click_event()
        else:
            self.info.value = "<h2> Đã hoàn thành tất cả!</h2>"

labeler = FullManualLabeler(BASE_DIR, CATEGORIES_MAP)

HTML(value="<script>\n        (function() {\n            const findAndBind = () => {\n                const im…

HTML(value="<script>setTimeout(() => { google.colab.kernel.invokeFunction('notebook.auto_next', [], {}); }, 10…

HTML(value="<script>\n        (function() {\n            const findAndBind = () => {\n                const im…

HTML(value="<script>setTimeout(() => { google.colab.kernel.invokeFunction('notebook.auto_next', [], {}); }, 10…

HTML(value="<script>\n        (function() {\n            const findAndBind = () => {\n                const im…

HTML(value="<script>setTimeout(() => { google.colab.kernel.invokeFunction('notebook.auto_next', [], {}); }, 10…

HTML(value="<script>\n        (function() {\n            const findAndBind = () => {\n                const im…

In [ ]:
import shutil
import os

# Định nghĩa đường dẫn
SOURCE_LABELS = '/content/labels'
TARGET_LABELS = '/content/labels_standard'

print(f" Đang chuyển đổi nhãn sang: {TARGET_LABELS} (giữ nguyên cấu trúc thư mục)...")

count = 0
# Duyệt qua các thư mục con (Chó, Mèo)
for root, dirs, files in os.walk(SOURCE_LABELS):
    for file in files:
        if file.endswith('.txt'):
            # Lấy đường dẫn file nguồn
            source_file = os.path.join(root, file)

            # Xác định thư mục con tương ứng (ví dụ: Chó hoặc Mèo)
            rel_path = os.path.relpath(root, SOURCE_LABELS)
            target_dir = os.path.join(TARGET_LABELS, rel_path)

            # Tạo thư mục con đích nếu chưa có
            os.makedirs(target_dir, exist_ok=True)

            # Copy file vào thư mục con tương ứng
            target_file = os.path.join(target_dir, file)
            shutil.copy2(source_file, target_file)
            count += 1

print(f" Đã chuyển xong {count} file nhãn vào {TARGET_LABELS} theo cấu trúc thư mục!")

 Đang chuyển đổi nhãn sang: /content/labels_standard (giữ nguyên cấu trúc thư mục)...
 Đã chuyển xong 11 file nhãn vào /content/labels_standard theo cấu trúc thư mục!


In [8]:
import os
from PIL import Image

# Cấu hình đường dẫn
IMAGE_BASE_DIR = '/content/drive/MyDrive/Data clean'
SOURCE_LABELS = '/content/labels'
TARGET_LABELS = '/content/labels_standard'

os.makedirs(TARGET_LABELS, exist_ok=True)

print(f" Bắt đầu chuyển đổi nhãn sang tọa độ pixel tuyệt đối...")

converted_count = 0

# Duyệt qua các thư mục con trong SOURCE_LABELS (Chó, Mèo)
for cat_name in os.listdir(SOURCE_LABELS):
    source_cat_dir = os.path.join(SOURCE_LABELS, cat_name)
    if not os.path.isdir(source_cat_dir): continue

    target_cat_dir = os.path.join(TARGET_LABELS, cat_name)
    os.makedirs(target_cat_dir, exist_ok=True)

    # Thư mục ảnh tương ứng trên Drive
    img_cat_dir = os.path.join(IMAGE_BASE_DIR, cat_name)

    for label_file in os.listdir(source_cat_dir):
        if not label_file.endswith('.txt'): continue

        label_path = os.path.join(source_cat_dir, label_file)
        # Tìm file ảnh tương ứng (thử các đuôi .jpg, .png...)
        img_name_no_ext = os.path.splitext(label_file)[0]
        img_path = None
        for ext in ['.jpg', '.jpeg', '.png', '.webp']:
            tmp_path = os.path.join(img_cat_dir, img_name_no_ext + ext)
            if os.path.exists(tmp_path):
                img_path = tmp_path
                break

        if img_path is None:
            print(f" Không tìm thấy ảnh cho nhãn: {label_file}")
            continue

        try:
            # Lấy kích thước ảnh
            with Image.open(img_path) as img:
                W, H = img.size

            # Đọc tọa độ YOLO và chuyển đổi
            new_lines = []
            with open(label_path, 'r') as f:
                for line in f:
                    parts = line.strip().split()
                    if len(parts) != 5: continue

                    cls, x_c, y_c, w_norm, h_norm = map(float, parts)

                    # Chuyển sang xmin, ymin, xmax, ymax (tuyệt đối)
                    xmin = (x_c - w_norm / 2) * W
                    ymin = (y_c - h_norm / 2) * H
                    xmax = (x_c + w_norm / 2) * W
                    ymax = (y_c + h_norm / 2) * H

                    # Định dạng: class xmin ymin xmax ymax
                    new_lines.append(f"{int(cls)} {xmin:.2f} {ymin:.2f} {xmax:.2f} {ymax:.2f}")

            # Lưu file mới
            with open(os.path.join(target_cat_dir, label_file), 'w') as f:
                f.write("\n".join(new_lines))

            converted_count += 1
        except Exception as e:
            print(f"Lỗi xử lý {label_file}: {e}")

print(f"\n Hoàn thành! Đã chuyển đổi {converted_count} file nhãn sang tọa độ pixel.")

 Bắt đầu chuyển đổi nhãn sang tọa độ pixel tuyệt đối...

 Hoàn thành! Đã chuyển đổi 3 file nhãn sang tọa độ pixel.
